# Phase 2: Statistics & Hypothesis Testing

**Overarching question:** What factors are associated with inspection failures and higher violation counts among Chicago food establishments?

## Hypotheses
1. Risk category ↔ Result → chi-square
2. Violation count across Facility Types → ANOVA (check normality → Kruskal-Wallis if needed)
3. Winter vs. rest-of-year pass rate → two-proportion z-test
4. Year ↔ avg. violations per inspection → correlation (check linearity → Spearman if needed)
5. Repeat-violation history ↔ Violation count → chi-square

In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect('chicago_food_safety.db')

In [4]:
h1=pd.read_sql_query("""
SELECT Risk, Results
FROM Inspections
WHERE Results IN ('Pass', 'Fail', 'Pass w/ Conditions')
    AND "Risk" IN ('Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)')
""",conn)
contingency_h1=pd.crosstab(h1['Risk'],h1['Results'])
contingency_h1

Results,Fail,Pass,Pass w/ Conditions
Risk,,,
Risk 1 (High),44764,124127,36306
Risk 2 (Medium),10602,28200,8303
Risk 3 (Low),5073,10370,2066


In [7]:
h2 = pd.read_sql_query("""
    SELECT i."Inspection ID", i."Facility Type",
           COUNT(v."Violations_code") AS violation_count
    FROM "Inspections" i
    LEFT JOIN "violations" v ON i."Inspection ID" = v."Inspection ID"
    WHERE i."Facility Type" IS NOT NULL
    GROUP BY i."Inspection ID", i."Facility Type"
    ORDER BY i."Inspection ID"
""", conn)
top_facility=h2["Facility Type"].value_counts().head(6).index
h2_top=h2[h2["Facility Type"].isin(top_facility)].copy()
h2_top

,Inspection ID,Facility Type,violation_count
0,44247,Restaurant,6
1,44248,Restaurant,3
2,44249,Restaurant,1
5,44252,Restaurant,3
6,44253,Restaurant,4
...,...,...,...
308997,2642298,Grocery Store,0
308998,2642300,School,4
308999,2642303,Daycare Above and Under 2 Years,5
309000,2642305,Grocery Store,0
